In [2]:
import torch
from wearable_affect.data import PROJECT_ROOT
from wearable_affect.third_party.papagei_resnet import ResNet1DMoE

model = ResNet1DMoE(
    in_channels=1, base_filters=32, kernel_size=3, stride=2,
    groups=1, n_block=18, n_classes=512, n_experts=3,
)

state = torch.load(PROJECT_ROOT / "weights" / "papagei_s.pt", map_location="cpu")
state = {k.removeprefix("module."): v for k, v in state.items()}
print(model.load_state_dict(state))

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()
print("device:", device)
print("parameters:", sum(p.numel() for p in model.parameters()))

<All keys matched successfully>
device: cuda
parameters: 5785612


In [3]:
x = torch.randn(6, 1, 1250, device=device)  # 6 segments, 1 channel, 10 s at 125 Hz

with torch.inference_mode():
    outputs = model(x)

print(len(outputs), outputs[0].shape)

4 torch.Size([6, 512])
